# 3.2 — RDD and DataFrame, Side by Side

**Chapter 3, sections 3.1 and 3.3.**

**The question this notebook answers:** the chapter asserts that every structured operation has
an RDD equivalent, that the structured form is shorter, and that it is faster for two
compounding reasons. What does that correspondence actually look like, operation by operation?

The notebook is built as **twelve pairs**. Each one states a task in one sentence, then gives
the RDD implementation and the DataFrame implementation in the same cell, one directly above the
other, and checks that the two produce the same answer. Nothing is asserted that is not run.

Read each cell top to bottom and the difference is hard to miss:

* the RDD version says **how** — key the records, carry an accumulator, unpack the tuple, divide
  at the end;
* the DataFrame version says **what** — group by this, average that.

Every RDD version below also ships a Python function to the executors. Every DataFrame version
below compiles to JVM code and never crosses into Python at all. The last section puts a number
on what that costs.

One table throughout: ten coffee ratings, small enough that every answer can be read in full.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-3.2")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")   # keep printed output clean
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# The SparkContext comes from the session -- half of this notebook is deliberately
# written against the RDD API, and `sc` is how you reach it.  There is no `sqlContext`
# in Spark 2.0 and later; code that uses one predates the SparkSession entirely.
sc = spark.sparkContext

print("Spark", spark.version)

Spark 4.2.0


## The data, in both representations

The same Python list, made into an RDD and into a DataFrame. That is the fairest possible
starting point: from here on, any difference between the two columns of code is a difference
between the APIs and not between the inputs.

In [2]:
RATINGS = [("Chris", "Espresso",   5), ("Chris", "Latte",     7),
           ("Chris", "Cold Brew",  7), ("Chris", "Flat White", 9),
           ("Peter", "Espresso",   8), ("Peter", "Latte",      9),
           ("Peter", "Cold Brew",  3),
           ("John",  "Espresso",   6), ("John",  "Latte",      4),
           ("John",  "Cold Brew",  6)]

ORIGINS = [("Espresso", "Italy"), ("Latte", "Italy"),
           ("Cold Brew", "Japan"), ("Flat White", "Australia")]

# ---- RDD ------------------------------------------------------------------
ratings_rdd = sc.parallelize(RATINGS)
origins_rdd = sc.parallelize(ORIGINS)

# ---- DataFrame ------------------------------------------------------------
ratings_df = spark.createDataFrame(RATINGS, ["taster", "coffee", "score"])
origins_df = spark.createDataFrame(ORIGINS, ["coffee", "origin"])

print("RDD       :", ratings_rdd.take(2), "   <- anonymous tuples, no names, no types")
print("DataFrame :")
ratings_df.show(2)
ratings_df.printSchema()

RDD       : [('Chris', 'Espresso', 5), ('Chris', 'Latte', 7)]    <- anonymous tuples, no names, no types
DataFrame :


+------+--------+-----+
|taster|  coffee|score|
+------+--------+-----+
| Chris|Espresso|    5|
| Chris|   Latte|    7|
+------+--------+-----+
only showing top 2 rows
root
 |-- taster: string (nullable = true)
 |-- coffee: string (nullable = true)
 |-- score: long (nullable = true)



There is the whole of the chapter's argument in two outputs. The RDD holds tuples: Spark knows
it has ten Python objects and nothing else about them. The DataFrame holds the same ten records
*plus a schema* — three named columns with declared types — and that is the entire difference
from which everything below follows.

## The checker

Every pair ends with a call to `compare`, which normalises both answers and **fails the cell if
they disagree**. It is what makes this notebook a demonstration rather than an assertion: if an
RDD version and a DataFrame version below ever stopped computing the same thing, the notebook
would stop running.

In [3]:
def compare(rdd_rows, df, round_to=3):
    """Check that an RDD answer and a DataFrame answer are the same, then print it once."""
    def norm(rows):
        out = []
        for r in rows:
            t = tuple(r) if isinstance(r, (tuple, list)) or hasattr(r, "asDict") else (r,)
            out.append(tuple(round(v, round_to) if isinstance(v, float) else v for v in t))
        return sorted(out, key=lambda t: tuple(str(x) for x in t))

    a, b = norm(rdd_rows), norm(df.collect())
    assert a == b, f"the two APIs disagree\n  RDD       : {a}\n  DataFrame : {b}"
    print(f"identical: yes   ({len(a)} row{'' if len(a) == 1 else 's'})")
    for row in a:
        print("   ", row)

---

## The pairs

### 1. Project — keep two of the three columns

`select` is the relational name for choosing columns. The RDD has no notion of a column, so
choosing one means writing a function that rebuilds the tuple, and Spark cannot see which fields
that function touched — it must read all three regardless.

In [4]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = ratings_rdd.map(lambda r: (r[0], r[2])).collect()

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.select("taster", "score")

compare(rdd_ans, df_ans)

identical: yes   (10 rows)
    ('Chris', 5)
    ('Chris', 7)
    ('Chris', 7)
    ('Chris', 9)
    ('John', 4)
    ('John', 6)
    ('John', 6)
    ('Peter', 3)
    ('Peter', 8)
    ('Peter', 9)


### 2. Filter — keep the ratings of 7 or more

The two look alike, and are not. `filter` on an RDD takes a Python function that is shipped to
every executor and called once per record. `where` on a DataFrame takes a *column expression*,
which is data the optimizer can read, move, and compile — and which it can push down to the file
reader, as notebook [3.4](03.04%20Parquet%20vs%20CSV.ipynb) measures.

In [5]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = ratings_rdd.filter(lambda r: r[2] >= 7).collect()

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.where(F.col("score") >= 7)

compare(rdd_ans, df_ans)

identical: yes   (5 rows)
    ('Chris', 'Cold Brew', 7)
    ('Chris', 'Flat White', 9)
    ('Chris', 'Latte', 7)
    ('Peter', 'Espresso', 8)
    ('Peter', 'Latte', 9)


### 3. Derive a column — band each rating as high or low

`withColumn` adds a column and leaves the rest of the row alone. The RDD version has to write
out every field it wants to keep, because a tuple has no concept of "the other columns" — which
is also why an RDD pipeline breaks whenever a field is inserted upstream.

In [6]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = (ratings_rdd
           .map(lambda r: (r[0], r[1], r[2], "high" if r[2] >= 7 else "low"))
           .collect())

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.withColumn(
    "band", F.when(F.col("score") >= 7, "high").otherwise("low"))

compare(rdd_ans, df_ans)

identical: yes   (10 rows)
    ('Chris', 'Cold Brew', 7, 'high')
    ('Chris', 'Espresso', 5, 'low')
    ('Chris', 'Flat White', 9, 'high')
    ('Chris', 'Latte', 7, 'high')
    ('John', 'Cold Brew', 6, 'low')
    ('John', 'Espresso', 6, 'low')
    ('John', 'Latte', 4, 'low')
    ('Peter', 'Cold Brew', 3, 'low')
    ('Peter', 'Espresso', 8, 'high')
    ('Peter', 'Latte', 9, 'high')


### 4. Distinct — which coffees appear at all

Same operator name on both sides, and the same shuffle underneath. This is the pair where the
two APIs are closest — and note that even here the RDD version needed a `map` first, because
"the coffee column" is not something an RDD can name.

In [7]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = ratings_rdd.map(lambda r: r[1]).distinct().collect()

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.select("coffee").distinct()

compare(rdd_ans, df_ans)

identical: yes   (4 rows)
    ('Cold Brew',)
    ('Espresso',)
    ('Flat White',)
    ('Latte',)


### 5. Count per key — how many ratings each coffee received

The canonical pair-RDD idiom: attach a 1 to every record and add the ones up. `groupBy().count()`
says the same thing in three words, and Spark performs the map-side combine on the reader's
behalf — the same optimization `reduceByKey` gets, but without the reader having to know that
`reduceByKey` is the one to reach for and `groupByKey` is not.

In [8]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = (ratings_rdd
           .map(lambda r: (r[1], 1))
           .reduceByKey(lambda a, b: a + b)
           .collect())

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.groupBy("coffee").count()

compare(rdd_ans, df_ans)

identical: yes   (4 rows)
    ('Cold Brew', 3)
    ('Espresso', 3)
    ('Flat White', 1)
    ('Latte', 3)


### 6. Average per key — the sharpest pair in the notebook

`reduceByKey` combines two values of the same shape, and an average is not of that shape. So the
RDD version has to carry a *(sum, count)* accumulator through the whole reduction and divide at
the very end. Three operators and a nested tuple to express one word.

This is the concrete form of the chapter's claim that the RDD is opaque: nothing in that code
tells Spark it is computing a mean. The DataFrame version names the operation, so Spark knows.

In [9]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = (ratings_rdd
           .map(lambda r: (r[1], (r[2], 1)))                    # (coffee, (score, 1))
           .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
           .mapValues(lambda t: t[0] / t[1])
           .collect())

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.groupBy("coffee").agg(F.avg("score").alias("avg_score"))

compare(rdd_ans, df_ans)

identical: yes   (4 rows)
    ('Cold Brew', 5.333)
    ('Espresso', 6.333)
    ('Flat White', 9.0)
    ('Latte', 6.667)


### 7. Several aggregates at once — count, mean and maximum per coffee

Every aggregate added to the RDD version widens the accumulator tuple and lengthens the
combiner. The DataFrame version adds one argument, and Spark still makes a single pass.

In [10]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = (ratings_rdd
           .map(lambda r: (r[1], (1, r[2], r[2])))              # (n, sum, max)
           .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], max(a[2], b[2])))
           .map(lambda kv: (kv[0], kv[1][0], kv[1][1] / kv[1][0], kv[1][2]))
           .collect())

# ---- DataFrame ------------------------------------------------------------
df_ans = (ratings_df.groupBy("coffee")
          .agg(F.count("*").alias("n"),
               F.avg("score").alias("mean"),
               F.max("score").alias("best")))

compare(rdd_ans, df_ans)

identical: yes   (4 rows)
    ('Cold Brew', 3, 5.333, 7)
    ('Espresso', 3, 6.333, 8)
    ('Flat White', 1, 9.0, 9)
    ('Latte', 3, 6.667, 9)


### 8. Sort and take the top three

`takeOrdered` is an action that returns a Python list to the driver; `orderBy().limit()` is a
pair of *transformations* that still describe a distributed result. The difference matters at
scale: the DataFrame form can be written back to storage or joined against, while the RDD form
has already left the cluster.

In [11]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = ratings_rdd.takeOrdered(3, key=lambda r: (-r[2], r[0], r[1]))

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.orderBy(F.desc("score"), "taster", "coffee").limit(3)

compare(rdd_ans, df_ans)

identical: yes   (3 rows)
    ('Chris', 'Flat White', 9)
    ('Peter', 'Espresso', 8)
    ('Peter', 'Latte', 9)


### 9. Join — attach each rating to its coffee's origin

An RDD join is a join of *pair* RDDs, so both sides must first be keyed by hand, and the result
arrives as a nested tuple that has to be unpacked. On the DataFrame the shared column name is
the entire join condition, and the result is a flat table.

In [12]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = (ratings_rdd
           .map(lambda r: (r[1], (r[0], r[2])))                 # key by coffee
           .join(origins_rdd)                                   # (coffee, ((taster, score), origin))
           .map(lambda kv: (kv[0], kv[1][0][0], kv[1][0][1], kv[1][1]))
           .collect())

# ---- DataFrame ------------------------------------------------------------
df_ans = ratings_df.join(origins_df, on="coffee").select(
    "coffee", "taster", "score", "origin")

compare(rdd_ans, df_ans)

identical: yes   (10 rows)
    ('Cold Brew', 'Chris', 7, 'Japan')
    ('Cold Brew', 'John', 6, 'Japan')
    ('Cold Brew', 'Peter', 3, 'Japan')
    ('Espresso', 'Chris', 5, 'Italy')
    ('Espresso', 'John', 6, 'Italy')
    ('Espresso', 'Peter', 8, 'Italy')
    ('Flat White', 'Chris', 9, 'Australia')
    ('Latte', 'Chris', 7, 'Italy')
    ('Latte', 'John', 4, 'Italy')
    ('Latte', 'Peter', 9, 'Italy')


### 10. Set operations — union, difference, intersection

The names match; the semantics differ in one respect worth knowing. `RDD.union` concatenates
elements, while `DataFrame.union` matches columns **by position**, not by name — so two frames
with the same column names in a different order will silently union wrongly. `unionByName` is
the safe form.

In [13]:
chris = ratings_rdd.filter(lambda r: r[0] == "Chris")
peter = ratings_rdd.filter(lambda r: r[0] == "Peter")
chris_df = ratings_df.where(F.col("taster") == "Chris")
peter_df = ratings_df.where(F.col("taster") == "Peter")

# ---- RDD ------------------------------------------------------------------
rdd_union  = chris.union(peter).collect()
rdd_coffee = (chris.map(lambda r: r[1]).subtract(peter.map(lambda r: r[1]))).collect()

# ---- DataFrame ------------------------------------------------------------
df_union  = chris_df.union(peter_df)
df_coffee = chris_df.select("coffee").subtract(peter_df.select("coffee"))

print("union:")
compare(rdd_union, df_union)
print("\ncoffees Chris rated that Peter did not:")
compare(rdd_coffee, df_coffee)

union:
identical: yes   (7 rows)
    ('Chris', 'Cold Brew', 7)
    ('Chris', 'Espresso', 5)
    ('Chris', 'Flat White', 9)
    ('Chris', 'Latte', 7)
    ('Peter', 'Cold Brew', 3)
    ('Peter', 'Espresso', 8)
    ('Peter', 'Latte', 9)

coffees Chris rated that Peter did not:


identical: yes   (1 row)
    ('Flat White',)


### 11. Group into a list — every score each taster gave

`groupByKey` is the RDD operator the previous chapter warned about: it moves every value across
the network before any combining happens. `collect_list` expresses the same result as an
aggregate function, which is both shorter and, being a named operation, something the optimizer
can reason about.

In [14]:
# ---- RDD ------------------------------------------------------------------
rdd_ans = (ratings_rdd
           .map(lambda r: (r[0], r[2]))
           .groupByKey()
           .mapValues(lambda vs: sorted(vs))
           .collect())

# ---- DataFrame ------------------------------------------------------------
df_ans = (ratings_df.groupBy("taster")
          .agg(F.sort_array(F.collect_list("score")).alias("scores")))

compare(rdd_ans, df_ans)

identical: yes   (3 rows)
    ('Chris', [5, 7, 7, 9])
    ('John', [4, 6, 6])
    ('Peter', [3, 8, 9])


### 12. Null handling — a column the RDD cannot help with

This pair has no clean RDD counterpart, which is itself the point. `dropna` is meaningful
because the DataFrame knows which column is which; the RDD version is an index into a tuple, and
it will read wrongly the day a field is added upstream.

In [15]:
WITH_GAPS = [("Chris", "Espresso", 5), ("Peter", "Latte", None), ("John", "Mocha", 6)]

# ---- RDD ------------------------------------------------------------------
rdd_ans = sc.parallelize(WITH_GAPS).filter(lambda r: r[2] is not None).collect()

# ---- DataFrame ------------------------------------------------------------
df_ans = spark.createDataFrame(WITH_GAPS, ["taster", "coffee", "score"]).dropna(subset=["score"])

compare(rdd_ans, df_ans)

identical: yes   (2 rows)
    ('Chris', 'Espresso', 5)
    ('John', 'Mocha', 6)


### 13. What the engine can see

Both APIs will show you what they are about to do. What they have to show is not comparable.

In [16]:
chain_rdd = (ratings_rdd
             .map(lambda r: (r[1], (r[2], 1)))
             .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))

print("=== RDD: toDebugString ===")
print(chain_rdd.toDebugString().decode())

=== RDD: toDebugString ===
(18) PythonRDD[139] at RDD at PythonRDD.scala:59 []
 |   MapPartitionsRDD[138] at mapPartitions at PythonRDD.scala:197 []
 |   ShuffledRDD[137] at partitionBy at DirectMethodHandleAccessor.java:103 []
 +-(18) PairwiseRDD[136] at reduceByKey at /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/ipykernel_92189/2942883089.py:3 []
    |   PythonRDD[135] at reduceByKey at /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/ipykernel_92189/2942883089.py:3 []
    |   ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:326 []


In [17]:
print("=== DataFrame: explain() ===")
ratings_df.groupBy("coffee").agg(F.avg("score")).explain()

=== DataFrame: explain() ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[coffee#1], functions=[avg(score#2L)])
   +- Exchange hashpartitioning(coffee#1, 200), ENSURE_REQUIREMENTS, [plan_id=592]
      +- HashAggregate(keys=[coffee#1], functions=[partial_avg(score#2L)])
         +- Project [coffee#1, score#2L]
            +- Scan ExistingRDD[taster#0,coffee#1,score#2L]




The RDD lineage names the *stages* — `MapPartitionsRDD`, `ShuffledRDD` — and says nothing
whatever about what any of them computes. Spark can schedule that graph faithfully and cannot
reason about it, because the only description it has of the work is a pointer to a Python
function.

The DataFrame plan names the *operations*: `HashAggregate` with `partial_avg` beneath the
exchange and `avg` above it. Spark inserted that partial aggregation itself, because it knows
what an average is and knows the operation is associative. That is what the schema bought.

---

## What the difference costs

Ten rows cannot show a performance gap. This section repeats **pair 6**, the average per key, on
four million rows read from a Parquet file — the same source for both, read afresh on every run,
so that neither side gets a cached head start.

In [18]:
BENCH = os.path.join(SCRATCH, "ch03-bench")
N = 4_000_000

if not os.path.exists(BENCH):
    (spark.range(N)
          .withColumn("coffee", F.concat(F.lit("C"), (F.rand(1) * 8).cast("int").cast("string")))
          .withColumn("score",  F.rand(2) * 10)
          .write.mode("overwrite").parquet(BENCH))

source = spark.read.parquet(BENCH)
print(f"{source.count():,} rows")

4,000,000 rows


In [19]:
def dataframe_way():
    rows = source.groupBy("coffee").agg(F.avg("score").alias("avg")).collect()
    return sorted((r["coffee"], round(r["avg"], 3)) for r in rows)

def rdd_way():
    rows = (source.rdd                                   # every row crosses into Python here
            .map(lambda r: (r["coffee"], (r["score"], 1)))
            .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
            .mapValues(lambda t: t[0] / t[1])
            .collect())
    return sorted((k, round(v, 3)) for k, v in rows)

def best_of(fn, runs=3):
    times = []
    for _ in range(runs):
        t0 = time.time()
        answer = fn()
        times.append(time.time() - t0)
    return min(times), answer

df_seconds,  df_answer  = best_of(dataframe_way)
rdd_seconds, rdd_answer = best_of(rdd_way)

print(f"same answer from both: {df_answer == rdd_answer}")
print(f"DataFrame : {df_seconds:6.2f} s")
print(f"RDD       : {rdd_seconds:6.2f} s   ->  {rdd_seconds / df_seconds:.1f}x slower")
print("\n(best of three runs, on one laptop; the ratio travels, the seconds do not)")

same answer from both: True
DataFrame :   0.08 s
RDD       :   0.99 s   ->  13.2x slower

(best of three runs, on one laptop; the ratio travels, the seconds do not)


The chapter says the gap "is so often not a matter of a few percent but of an order of
magnitude", and here it is, on an identical answer.

Two independent causes compound, and the chapter is careful to separate them:

1. **The plan is optimized.** Spark knows the DataFrame version computes an average, so it
   inserts a partial aggregation before the shuffle and moves one partial result per key per
   partition instead of every record. It cannot do this for the RDD version, because it cannot
   see inside the lambdas.
2. **Execution stays inside the JVM.** The DataFrame version never enters Python. The RDD
   version must start a Python process beside every executor and serialize every one of four
   million rows across the boundary — the line marked in `rdd_way` above — and serialize the
   results back.

Neither cause depends on the RDD version being written badly. It is written the way the previous
chapter recommends, with `reduceByKey` rather than `groupByKey`, and it is still an order of
magnitude behind.

---

## The correspondence, in one table

| Task | RDD | DataFrame |
|------|-----|-----------|
| choose columns | `map(lambda r: (r[0], r[2]))` | `select("taster", "score")` |
| choose rows | `filter(lambda r: r[2] >= 7)` | `where(col("score") >= 7)` |
| add a column | `map` rebuilding the whole tuple | `withColumn("band", …)` |
| distinct | `map(...).distinct()` | `select("coffee").distinct()` |
| count per key | `map(...).reduceByKey(add)` | `groupBy("coffee").count()` |
| average per key | `map` → `reduceByKey` on *(sum, count)* → `mapValues` | `groupBy("coffee").avg("score")` |
| several aggregates | one wider accumulator tuple | `agg(count, avg, max)` |
| top *k* | `takeOrdered(k, key=…)` | `orderBy(desc(…)).limit(k)` |
| join | key both sides, join, unpack the nested tuple | `join(other, on="coffee")` |
| union / difference | `union` / `subtract` | `union` (**by position**) / `subtract` |
| group into a list | `groupByKey().mapValues(list)` | `agg(collect_list(…))` |
| drop nulls | `filter(lambda r: r[2] is not None)` | `dropna(subset=["score"])` |
| see what will run | `toDebugString()` — stage names | `explain()` — operator names |

## Conclusion

Twelve tasks, each computed twice, each checked to give the same answer. The RDD column is never
*wrong* — it is unexamined. Three things separate the two, and all three follow from the schema:

* **Length.** The average per key took three operators and a nested accumulator on the left, and
  one call on the right. The gap widens with every aggregate added.
* **Fragility.** Every RDD version above reaches its fields by position. Insert a column upstream
  and every one of them keeps running and returns nonsense. The DataFrame versions name their
  columns and fail loudly instead.
* **Speed.** The order of magnitude measured above on four million rows, from two compounding
  causes: the plan is optimized because Spark understands the operation, and execution never
  leaves the JVM because no Python function is involved.

The RDD remains the abstraction to understand — it is what the DataFrame is built on, and the
previous chapter developed it for that reason. It is simply not, in most cases, the abstraction
to program against. Where this goes next:
[3.6](03.06%20Catalyst%20AQE%20and%20Skew.ipynb) opens up the plans this notebook only glanced
at, and shows what Catalyst does with the freedom the schema gives it.